# Piece 2 - joint sentence + token head

A second output head on the shared BiLSTM predicts whether the whole tweet is
offensive:

    loss = token_loss + lambda * sentence_loss

Lambda is swept over 0, 0.1, 0.3, 0.5, 1.0. Lambda 0 builds the model with no
sentence head, so it is the control for this sweep.

Scored on validation over 5 seeds. Test is not scored here.

## How to run

1. Settings: Accelerator = GPU T4 x2, Internet = On
2. Set `OWNER` below
3. Leave `SMOKE = True` and Run All to check the setup
4. Set `SMOKE = False`
5. Save Version -> Save & Run All (Commit)
6. Collect the two result files from the version's Output tab

## Settings

In [ ]:
OWNER  = 'YOUR_NAME'     # <-- EDIT
SMOKE  = True            # True = 1 seed / 3 epochs. False = the full sweep.

REPO   = 'https://github.com/hatheem-r/project_DNN.git'
BRANCH = 'main'

print('owner', OWNER, '| smoke', SMOKE)

## GPU

In [ ]:
import torch
print('torch', torch.__version__, '| CUDA', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')
assert torch.cuda.is_available(), 'Settings -> Accelerator -> GPU T4 x2'

## Code

In [ ]:
import os, shutil
os.chdir('/kaggle/working')
if os.path.exists('project'):
    shutil.rmtree('project')
!git clone -q -b $BRANCH $REPO project
os.chdir('/kaggle/working/project')
!pip install -q 'datasets<3.0.0' pytorch-crf sentencepiece 2>&1 | tail -1
print('cwd', os.getcwd())

## fastText vectors

About 460 MB, downloaded to `/kaggle/temp` so it is not saved as notebook
output. The script reads the path from `SOLD_VECTORS`.

In [ ]:
import os
os.makedirs('/kaggle/temp/embeddings', exist_ok=True)
VEC = '/kaggle/temp/embeddings/cc.si.300.vec.gz'
if not os.path.exists(VEC):
    !wget -q -O $VEC https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.si.300.vec.gz
os.environ['SOLD_VECTORS'] = VEC
os.environ['OWNER'] = OWNER
!ls -lh $VEC

## Tests

The alignment tests check that piece lists line up with words. A break there
shifts labels against tokens without raising an error.

In [ ]:
!python tests/test_metrics.py | tail -2
!python tests/test_subword_alignment.py | tail -2

## Run

Sweeps lambda over 0, 0.1, 0.3, 0.5, 1.0 at 5 seeds each. `--loss` is left at
its default, `cross_entropy`. Roughly 2.5 hours on a T4.

In [ ]:
import os, subprocess
os.makedirs('results', exist_ok=True)
os.makedirs('artifacts', exist_ok=True)

args = ['python', 'notebooks/09_pieces_234.py', '--piece2']
args += (['--seeds', '1', '--epochs', '3', '--patience', '2'] if SMOKE
         else ['--seeds', '1', '2', '3', '4', '5'])
print(' '.join(args), flush=True)

with open('results/piece2_report.txt', 'w') as log:
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
        log.write(line); log.flush()
    rc = p.wait()
print('exit code', rc)

## Save the output

In [ ]:
import os, shutil

if SMOKE:
    if os.path.exists('results/results_piece2.csv'):
        os.remove('results/results_piece2.csv')   # discard the smoke rows
    print('Setup OK. Set SMOKE = False, then Save Version -> Save & Run All.')
else:
    for f in ('results/results_piece2.csv', 'results/piece2_report.txt'):
        shutil.copy(f, '/kaggle/working/')
    print('saved to /kaggle/working')

## Report

From the ranking table at the end of the output:

- lambda = 0 (control): F1 and std
- best lambda, its F1, std, precision and recall
- the verdict line comparing the best lambda against lambda = 0
- the lambda value for Piece 4

Download the two files from the Output tab, put them in `results/`, then:

```
git add results/results_piece2.csv results/piece2_report.txt
git commit -m "piece2 results"
git pull --rebase
git push
```